# Modeling a Plant with Input

## Adding an External Input

The [previous notebook](plant.ipynb) modelled a pendulum evolving freely with no external influence. Real control systems have an **input** $u_k$ — a torque, force, or voltage that a controller can vary to change how the plant behaves.

In `dynamicalnodes`, adding an input is as simple as including `uk` in the signature of `f`:

$$x_{k+1} = f(x_k, u_k, \ldots)$$

`_smart_call()` dispatches `uk` to `f` automatically — no changes to `DynamicalSystem` itself are needed.

This notebook adds an **applied torque** $\tau_k$ to the pendulum. Because the pendulum is nonlinear, we use the same linearize-then-discretize pipeline from the previous notebook — now extended to handle an input term.

## Forced Pendulum

Applying an external torque $\tau_k$ at the pivot gives:

$$mL^2 \ddot{\theta} = -mgL\sin(\theta) - cL^2\dot{\theta} + \tau_k$$

Dividing through by $mL^2$:

$$\ddot{\theta} = -\frac{g}{L}\sin(\theta) - c\dot{\theta} + \frac{\tau_k}{mL^2}$$

With state $x = [\theta,\; \dot{\theta}]^\top$, the continuous-time model is $\dot{x} = f_{\text{ct}}(x, u)$:

$$
\dot{x} =
\begin{bmatrix} \dot{\theta} \\ -\dfrac{g}{L}\sin(\theta) - c\dot{\theta} + \dfrac{\tau}{mL^2} \end{bmatrix}
$$

![Pendulum with applied torque τ_k at the pivot, displacing the mass m by angle θ from vertical.](../../_static/figures/pendulum_torque.svg)

## Linearization and Discretization with Input

### Linearized Continuous-Time Model

Applying $\sin(\theta) \approx \theta$ around $\theta = 0$ yields a linear continuous-time system:

$$\dot{x} = Ax + B\tau, \qquad
A = \begin{bmatrix} 0 & 1 \\ -g/L & -c \end{bmatrix},
\quad
B = \begin{bmatrix} 0 \\ 1/mL^2 \end{bmatrix}$$

The discrete-time form is $x_{k+1} = F x_k + G \tau_k$. As before, two discretization methods give different $(F, G)$ pairs.

### Method 1 — Forward Euler

Finite-difference approximation applied to both the state and input terms:

$$\boxed{F_{\text{Euler}} = I + \Delta t\, A, \qquad G_{\text{Euler}} = \Delta t\, B}$$

### Method 2 — Zero-Order Hold (ZOH)

Exact solution for a linear system driven by a piecewise-constant input. Both $F_{\text{ZOH}}$ and $G_{\text{ZOH}}$ are extracted from a single augmented matrix exponential:

$$
\exp\!\left(\begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix}\Delta t\right)
=
\begin{bmatrix} F_{\text{ZOH}} & G_{\text{ZOH}} \\ 0 & 1 \end{bmatrix}
$$

$$\boxed{F_{\text{ZOH}} = e^{A\Delta t}, \qquad G_{\text{ZOH}} = \int_0^{\Delta t} e^{A\tau} B\, d\tau}$$

The integral for $G_{\text{ZOH}}$ has no closed form in general, which is why the augmented-exponential trick is used in practice.

## Implementation

In [ ]:
import numpy as np
from scipy.linalg import expm
from scipy.integrate import solve_ivp
from dynamicalnodes import DynamicalSystem

### System Matrices

In [ ]:
g = 9.81  # m/s²
L = 1.0   # m
m = 1.0   # kg
c = 0.3   # damping coefficient (1/s)
dt = 0.15 # s  — same coarse timestep as previous notebook

sim_time = np.arange(0, 10, dt)

A = np.array([[0.0,    1.0],
              [-g / L, -c]])
B = np.array([0.0, 1.0 / (m * L**2)])

# Method 1: Euler
F_euler = np.eye(2) + dt * A
G_euler = dt * B

# Method 2: ZOH via augmented matrix exponential
M = np.zeros((3, 3))
M[:2, :2] = A
M[:2,  2] = B
M_exp = expm(M * dt)
F_zoh = M_exp[:2, :2]
G_zoh = M_exp[:2,  2]

print("G_euler:", G_euler)
print("G_zoh:  ", G_zoh)

### DynamicalSystem Blocks

`f` now includes `uk` in its signature. Both plants share the same structure; only `F` and `G` differ.

In [ ]:
def pendulum_f(xk, uk, F, G):
    return F @ xk + G * uk


def pendulum_h(xk):
    return xk[0]  # angle θ


euler_plant = DynamicalSystem(f=pendulum_f, h=pendulum_h)
zoh_plant   = DynamicalSystem(f=pendulum_f, h=pendulum_h)

### Nonlinear Reference

A helper that integrates the full nonlinear forced pendulum with `solve_ivp` for a given torque schedule.

In [ ]:
def nonlinear_reference(torque_fn):
    """Integrate the nonlinear pendulum ODE with a time-varying torque."""
    def ode(t, x):
        tau = torque_fn(t)
        return [x[1], -(g / L) * np.sin(x[0]) - c * x[1] + tau / (m * L**2)]

    sol = solve_ivp(
        ode,
        t_span=(sim_time[0], sim_time[-1]),
        y0=[0.0, 0.0],
        t_eval=sim_time,
        method="RK45",
        rtol=1e-9,
    )
    return sol.y[0]

### Simulation 1 — Step Torque

A constant torque $\tau = 3\,\text{N·m}$ is applied from rest. The pendulum accelerates, overshoots, then settles at the equilibrium angle $\theta^* = \tau/(mgL)$ predicted by the linearized model.

In [ ]:
tau_step = 3.0  # N·m
theta_eq = tau_step / (m * g * L)  # linearized equilibrium

xk_e, xk_z = np.zeros(2), np.zeros(2)
step_euler, step_zoh = [], []

for _ in sim_time:
    xk_e, th_e = euler_plant.step(xk=xk_e, uk=tau_step, F=F_euler, G=G_euler)
    xk_z, th_z = zoh_plant.step(  xk=xk_z, uk=tau_step, F=F_zoh,   G=G_zoh)
    step_euler.append(th_e)
    step_zoh.append(th_z)

step_true = nonlinear_reference(lambda t: tau_step)

### Simulation 2 — Sinusoidal Torque at Natural Frequency

Driving the pendulum at its natural frequency $\omega_n = \sqrt{g/L}$ excites a resonant response. The linearized models predict growing oscillations (for underdamped systems); damping eventually limits the amplitude.

In [ ]:
omega_n  = np.sqrt(g / L)  # rad/s
tau_amp  = 1.5             # N·m

xk_e, xk_z = np.zeros(2), np.zeros(2)
sin_euler, sin_zoh, sin_inputs = [], [], []

for tk in sim_time:
    uk = tau_amp * np.sin(omega_n * tk)
    xk_e, th_e = euler_plant.step(xk=xk_e, uk=uk, F=F_euler, G=G_euler)
    xk_z, th_z = zoh_plant.step(  xk=xk_z, uk=uk, F=F_zoh,   G=G_zoh)
    sin_euler.append(th_e)
    sin_zoh.append(th_z)
    sin_inputs.append(uk)

sin_true = nonlinear_reference(lambda t: tau_amp * np.sin(omega_n * t))

### Results

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(13, 7), sharex=True)

kw_true  = dict(color="black",     linewidth=2,   label="Nonlinear (RK45)")
kw_euler = dict(color="tomato",    linewidth=1.5, linestyle="--", label="Euler")
kw_zoh   = dict(color="steelblue", linewidth=1.5, linestyle="-.", label="ZOH")

# ── Step torque ─────────────────────────────────────────────────────────────
axs[0, 0].plot(sim_time, step_true,  **kw_true)
axs[0, 0].plot(sim_time, step_euler, **kw_euler)
axs[0, 0].plot(sim_time, step_zoh,   **kw_zoh)
axs[0, 0].axhline(theta_eq, color="gray", linewidth=0.8, linestyle=":",
                  label=f"linearized eq.  $\\theta^* = {theta_eq:.3f}$ rad")
axs[0, 0].set_ylabel("Angle $\\theta_k$ (rad)")
axs[0, 0].set_title("Step Torque — Angle")
axs[0, 0].legend(fontsize=8)

axs[1, 0].axhline(tau_step, color="tomato", linewidth=1.5)
axs[1, 0].set_ylabel("Torque $\\tau_k$ (N·m)")
axs[1, 0].set_xlabel("Time (s)")
axs[1, 0].set_title("Step Torque — Input")
axs[1, 0].set_ylim(-0.5, tau_step + 1.5)

# ── Sinusoidal torque at resonance ──────────────────────────────────────────
axs[0, 1].plot(sim_time, sin_true,  **kw_true)
axs[0, 1].plot(sim_time, sin_euler, **kw_euler)
axs[0, 1].plot(sim_time, sin_zoh,   **kw_zoh)
axs[0, 1].set_ylabel("Angle $\\theta_k$ (rad)")
axs[0, 1].set_title(f"Resonant Sinusoidal Torque — Angle  ($\\omega = \\omega_n = {omega_n:.2f}$ rad/s)")
axs[0, 1].legend(fontsize=8)

axs[1, 1].plot(sim_time, sin_inputs, color="tomato", linewidth=1.5)
axs[1, 1].set_ylabel("Torque $\\tau_k$ (N·m)")
axs[1, 1].set_xlabel("Time (s)")
axs[1, 1].set_title("Resonant Sinusoidal Torque — Input")

plt.suptitle(f"Forced Pendulum: Euler vs ZOH vs Nonlinear Reference  ($\\Delta t = {dt}$ s)",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## Summary

Adding an input $u_k$ to a `DynamicalSystem` requires two things:

1. Include `uk` in `f`'s parameter list — `_smart_call()` routes it automatically.
2. Discretize the input matrix $B$ alongside $A$ into $G$:

| Method | $F$ | $G$ |
|---|---|---|
| Euler | $I + \Delta t A$ | $\Delta t B$ |
| ZOH | $e^{A\Delta t}$ | $\int_0^{\Delta t} e^{A\tau} B\, d\tau$ (via augmented expm) |

The ZOH $G$ accounts for how the input propagates through the continuous-time dynamics over the full interval $\Delta t$, not just at the endpoints. The difference between the two $G$ matrices is what drives the divergence seen in the plots above.

Both simulations here are **open-loop** — $\tau_k$ is computed ahead of time and does not react to $\theta_k$. The next notebook, [Modeling a Feedback System](plant_feedback_system.ipynb), closes the loop.